# Day 2 — Feature Engineering & Preprocessing

Goal: turn the cleaned Telco data into an **ML-ready** train/test dataset.

**Pipeline**
```
Cleaned CSV → Feature Engineering → Encoding/Scaling → Train/Test Split → Save
```

**Rules**
- Do **not** use `Churn` to create features (no target leakage)
- Fit the preprocessor **only** on training data
- Do **not** train a churn model yet (that is Day 3)

## 1. Setup and load cleaned Day 1 data

In [ ]:
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from preprocessing import (
    BINARY_FEATURES,
    CATEGORICAL_FEATURES,
    NUMERIC_FEATURES,
    build_preprocessor,
    create_features,
    get_feature_matrix,
    get_feature_summary,
)

CLEANED_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned_telco.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"

print("Project root:", PROJECT_ROOT)
print("Cleaned data exists:", CLEANED_PATH.exists())

In [ ]:
df = pd.read_csv(CLEANED_PATH)

print("Shape:", df.shape)
print("\nColumns:")
print(list(df.columns))
print("\nData types:")
print(df.dtypes)
print("\nFirst 5 rows:")
display(df.head())
print("\nMissing values:")
print(df.isnull().sum())
print("\nTotalCharges is numeric:", pd.api.types.is_numeric_dtype(df["TotalCharges"]))

## 2. Identify features

| Role | Columns |
|------|---------|
| Target | `Churn` (encode No=0, Yes=1) |
| Identifier | `customerID` |
| Numerical (base) | `SeniorCitizen`, `tenure`, `MonthlyCharges`, `TotalCharges` |
| Categorical (base) | `gender`, `Partner`, `Dependents`, service columns, `Contract`, `PaperlessBilling`, `PaymentMethod` |

### Why exclude `customerID`?
`customerID` only labels a person. It does not describe behavior.
If the model learns IDs, it memorizes customers instead of learning patterns, and it fails on new customers.

In [ ]:
id_col = "customerID"
target_col = "Churn"

base_numeric = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]
base_categorical = [
    c for c in df.columns
    if c not in base_numeric + [id_col, target_col]
]

print("Identifier:", id_col)
print("Target:", target_col)
print("Base numerical:", base_numeric)
print("Base categorical:", base_categorical)
print("\nTarget value counts (before encoding):")
print(df[target_col].value_counts())

## 3. Feature engineering

We create business features using only information known about the customer today.
We never use `Churn` here.

In [ ]:
df_feat = create_features(df)

engineered = [
    "tenure_group",
    "total_services",
    "has_security_service",
    "has_streaming_service",
    "is_month_to_month",
    "is_long_term_customer",
    "average_monthly_revenue",
]

print("New features:")
for f in engineered:
    print(" -", f)

display(
    df_feat[
        ["customerID", "tenure", "Contract", "MonthlyCharges", "TotalCharges"]
        + engineered
        + ["Churn"]
    ].head(10)
)

print("\nWhy these features help:")
print("1. tenure_group — new customers often churn more than long-tenure customers")
print("2. total_services — customers with more products may be stickier")
print("3. has_security_service — support/security add-ons can signal engagement")
print("4. has_streaming_service — entertainment bundle usage")
print("5. is_month_to_month — Day 1 showed this group has much higher churn")
print("6. is_long_term_customer — longer relationships tend to be more stable")
print("7. average_monthly_revenue — spend intensity without using future data")

## 4. Build X and y (encode target, drop customerID)

In [ ]:
X, y = get_feature_matrix(df_feat)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("customerID in X?", "customerID" in X.columns)
print("Churn in X?", "Churn" in X.columns)
print("\nTarget encoding check (should be 0/1):")
print(y.value_counts().sort_index())
print("\nFeature groups used by the preprocessor:")
print("Numeric (scaled):", NUMERIC_FEATURES)
print("Binary (passthrough):", BINARY_FEATURES)
print("Categorical (one-hot):", CATEGORICAL_FEATURES)

## 5. Why encoding and scaling?

- **Encoding:** most ML models need numbers. `OneHotEncoder` turns categories like `Contract=Month-to-month` into 0/1 columns. `handle_unknown="ignore"` safely handles rare/new categories at prediction time.
- **Scaling:** features like `TotalCharges` are much larger than `tenure`. `StandardScaler` puts continuous numbers on a similar scale so models are not dominated by large-magnitude columns.
- We do **not** scale one-hot categorical columns.

## 6. Train/test split FIRST (before fitting preprocessor)

Fitting on the full dataset before splitting would leak test-set information into scaling/encoding statistics.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,  # keep similar churn rate in train and test
)

print(f"Training features: {X_train.shape}")
print(f"Testing features:  {X_test.shape}")
print(f"Training target:   {y_train.shape}")
print(f"Testing target:    {y_test.shape}")

## 7. Class balance

Class imbalance matters because a model can look "accurate" by mostly predicting the majority class (No churn).
For Day 2 we only measure and document the imbalance — we do not artificially rebalance yet.

In [ ]:
def show_balance(name, series):
    counts = series.value_counts().sort_index()
    pct = (series.value_counts(normalize=True).sort_index() * 100).round(2)
    print(f"\n{name}")
    print("counts:\n", counts.rename({0: "No (0)", 1: "Yes (1)"}))
    print("percentages:\n", pct.rename({0: "No (0)", 1: "Yes (1)"}))

show_balance("Complete dataset", y)
show_balance("Training dataset", y_train)
show_balance("Testing dataset", y_test)

## 8. Fit preprocessing pipeline on TRAIN only, then transform train + test

In [ ]:
preprocessor = build_preprocessor()
preprocessor.fit(X_train)  # FIT ON TRAIN ONLY

X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

feature_names = list(preprocessor.get_feature_names_out())

print("Transformed training shape:", X_train_processed.shape)
print("Transformed testing shape:", X_test_processed.shape)
print("Number of ML-ready columns:", len(feature_names))
print("NaNs in train:", np.isnan(X_train_processed).sum())
print("NaNs in test:", np.isnan(X_test_processed).sum())
print("\nSample output feature names:")
print(feature_names[:15], "...")

## 9. Save preprocessor, feature columns, and processed data

We use `OneHotEncoder(sparse_output=False)`, so the result is a dense numeric matrix.
That means CSV is appropriate and easy for beginners.
(If we had kept sparse matrices, CSV would be a bad fit; we would save `.npz` / `.parquet` instead.)

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

X_train_df = pd.DataFrame(X_train_processed, columns=feature_names)
X_test_df = pd.DataFrame(X_test_processed, columns=feature_names)

X_train_df.to_csv(PROCESSED_DIR / "X_train.csv", index=False)
X_test_df.to_csv(PROCESSED_DIR / "X_test.csv", index=False)
y_train.to_csv(PROCESSED_DIR / "y_train.csv", index=False, header=["Churn"])
y_test.to_csv(PROCESSED_DIR / "y_test.csv", index=False, header=["Churn"])

# Raw (pre-transform) splits are useful for debugging
X_train.to_csv(PROCESSED_DIR / "X_train_raw.csv", index=False)
X_test.to_csv(PROCESSED_DIR / "X_test_raw.csv", index=False)
df_feat.to_csv(PROCESSED_DIR / "featured_telco.csv", index=False)

joblib.dump(preprocessor, MODELS_DIR / "preprocessor.pkl")
joblib.dump(
    {
        "input_features": list(X.columns),
        "output_features": feature_names,
        "numeric_features": NUMERIC_FEATURES,
        "binary_features": BINARY_FEATURES,
        "categorical_features": CATEGORICAL_FEATURES,
    },
    MODELS_DIR / "feature_columns.pkl",
)

summary = get_feature_summary()
summary.to_csv(REPORTS_DIR / "feature_summary.csv", index=False)

print("Saved:")
print(" -", MODELS_DIR / "preprocessor.pkl")
print(" -", MODELS_DIR / "feature_columns.pkl")
print(" -", PROCESSED_DIR / "X_train.csv")
print(" -", PROCESSED_DIR / "X_test.csv")
print(" -", PROCESSED_DIR / "y_train.csv")
print(" -", PROCESSED_DIR / "y_test.csv")
print(" -", REPORTS_DIR / "feature_summary.csv")

## 10. Feature summary table

In [ ]:
display(get_feature_summary())

## 11. Final verification

In [ ]:
loaded = joblib.load(MODELS_DIR / "preprocessor.pkl")
check = loaded.transform(X_test.head(5))

print("Reload OK. Transformed sample shape:", check.shape)
print("X_train.csv exists:", (PROCESSED_DIR / "X_train.csv").exists())
print("X_test.csv exists:", (PROCESSED_DIR / "X_test.csv").exists())
print("preprocessor.pkl exists:", (MODELS_DIR / "preprocessor.pkl").exists())

print("\nDay 2 verification passed.")
print("Ready for Day 3 churn ML models — but do not train them yet.")